# 📘 Audio Mental Health Detection Pipeline
Classical Machine Learning Pipeline for Mental Health Prediction from Speech

**Steps**
1. Install dependencies
2. Imports
3. Audio preprocessing
4. Audio visualization
5. Feature extraction
6. Feature visualization
7. Dataset creation
8. SMOTE balancing
9. Model training
10. Inference

---
# 1️⃣ Install Dependencies

In [ ]:
# %pip install numpy pandas matplotlib seaborn scikit-learn xgboost lightgbm librosa noisereduce tqdm praat-parselmouth ipython webrtcvad-wheels imbalanced-learn

---
# 2️⃣ Imports

In [34]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display
import IPython.display as ipd

import noisereduce as nr
import webrtcvad
import scipy.signal

import parselmouth
from parselmouth.praat import call

from pathlib import Path
from tqdm import tqdm

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import xgboost as xgb
import lightgbm as lgb

from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

---
# 3️⃣ Audio Preprocessing

This step performs:
* Noise reduction
* Voice activity detection
* RMS normalization
* Pre-emphasis filtering

In [35]:
def run_vad_on_audio(audio_array, sample_rate, aggressiveness=3):

    frame_duration_ms = 30
    frame_size = int(sample_rate * (frame_duration_ms / 1000))

    vad = webrtcvad.Vad(aggressiveness)

    audio_int16 = (audio_array * 32767).astype(np.int16)

    padding = frame_size - (len(audio_int16) % frame_size)

    if padding < frame_size:
        audio_int16 = np.pad(audio_int16, (0, padding), mode='constant')

    voiced_frames = []

    for i in range(0, len(audio_int16), frame_size):

        frame = audio_int16[i:i+frame_size]

        if len(frame) == frame_size and vad.is_speech(frame.tobytes(), sample_rate):
            voiced_frames.append(frame)

    if len(voiced_frames) == 0:
        return audio_array

    voiced_audio = np.concatenate(voiced_frames)

    return voiced_audio.astype(np.float32) / 32767

### RMS Normalization

In [36]:
def normalize_rms(audio, target_rms=0.1):

    rms = np.sqrt(np.mean(audio**2))

    if rms > 0:
        audio = audio * (target_rms / rms)

    return audio

### Pre-emphasis Filter

In [37]:
def apply_preemphasis(audio, coeff=0.97):

    return np.append(audio[0], audio[1:] - coeff * audio[:-1])

### Main Preprocessing Function

In [38]:
def preprocess_audio(file_path):

    sample_rate = 16000

    y, sr = librosa.load(file_path, sr=sample_rate)

    if len(y) == 0:
        return None, None, None

    y_denoised = nr.reduce_noise(y=y, sr=sr)

    y_vad = run_vad_on_audio(y_denoised, sr)

    y_norm = normalize_rms(y_vad)

    y_clean = apply_preemphasis(y_norm)

    return y_clean, y, sr

---
# 4️⃣ Audio Visualization
### Waveform

In [39]:
def visualize_audio(file):

    y_clean, y_orig, sr = preprocess_audio(file)

    plt.figure(figsize=(14,6))

    plt.subplot(2,1,1)
    librosa.display.waveshow(y_orig, sr=sr)
    plt.title("Original Audio")

    plt.subplot(2,1,2)
    librosa.display.waveshow(y_clean, sr=sr)
    plt.title("Cleaned Audio")

    plt.tight_layout()
    plt.show()

    print("Original Audio")
    ipd.display(ipd.Audio(y_orig, rate=sr))

    print("Cleaned Audio")
    ipd.display(ipd.Audio(y_clean, rate=sr))

### Spectrogram

In [40]:
def visualize_spectrogram(file):

    y_clean, y_orig, sr = preprocess_audio(file)

    plt.figure(figsize=(14,5))

    S = librosa.stft(y_clean)
    S_db = librosa.amplitude_to_db(abs(S))

    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz')

    plt.colorbar()
    plt.title("Spectrogram")
    plt.show()

---
# 5️⃣ Feature Extraction

Extracts **speech diagnostic features**.

In [41]:
def extract_audio_features(audio, sr):

    features = {}

    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)

    features["mfcc_mean"] = np.mean(mfcc)
    features["mfcc_std"] = np.std(mfcc)

    delta = librosa.feature.delta(mfcc)
    features["delta_mean"] = np.mean(delta)
    features["delta_std"] = np.std(delta)

    sound = parselmouth.Sound(audio, sr)
    pitch = sound.to_pitch()

    pitch_values = pitch.selected_array["frequency"]
    pitch_values = pitch_values[pitch_values>0]

    if len(pitch_values)>0:

        features["pitch_mean"] = np.mean(pitch_values)
        features["pitch_std"] = np.std(pitch_values)

    else:

        features["pitch_mean"] = 0
        features["pitch_std"] = 0

    rms = librosa.feature.rms(y=audio)[0]

    features["energy_mean"] = np.mean(rms)
    features["energy_std"] = np.std(rms)

    zcr = librosa.feature.zero_crossing_rate(audio)[0]

    features["zcr_mean"] = np.mean(zcr)

    return features

---
# 6️⃣ Feature Visualization

In [42]:
def visualize_features(file):

    y_clean, _, sr = preprocess_audio(file)

    feats = extract_audio_features(y_clean, sr)

    df = pd.DataFrame([feats]).T
    df.columns=["value"]

    plt.figure(figsize=(8,5))

    sns.barplot(x=df["value"], y=df.index)

    plt.title("Extracted Audio Features")

    plt.show()

---
# 7️⃣ Dataset Creation

In [43]:
def create_dataset(folder, label):

    rows=[]

    for file in tqdm(os.listdir(folder)):

        if file.endswith(".wav"):

            path=os.path.join(folder,file)

            y_clean,_,sr=preprocess_audio(path)

            feats=extract_audio_features(y_clean,sr)

            feats["label"]=label

            rows.append(feats)

    return pd.DataFrame(rows)

### Example

In [44]:
normal_df=create_dataset("../data/raw/audio/normal",0)
depressed_df=create_dataset("../data/raw/audio/depression",1)

df=pd.concat([normal_df,depressed_df])

df.head()

100%|██████████| 6/6 [00:41<00:00,  7.00s/it]


,mfcc_mean,mfcc_std,delta_mean,delta_std,pitch_mean,pitch_std,energy_mean,energy_std,zcr_mean,label
0,-37.180248,120.310211,-0.018338,9.964116,179.989018,117.235775,0.028378,0.030707,0.193886,1
1,-35.686832,115.228416,-0.046313,10.619451,172.779655,100.641604,0.036140,0.034367,0.233176,1
2,-37.221893,118.543098,0.000298,9.894571,194.253424,121.586580,0.030599,0.033752,0.231092,1
3,-37.873314,119.295692,-0.001196,10.418610,191.820835,127.731262,0.036012,0.041035,0.212860,1
4,-36.085762,110.859741,-0.015720,9.507435,177.067811,103.223317,0.027787,0.023375,0.199462,1


---
# 8️⃣ SMOTE Balancing

In [45]:
X=df.drop("label",axis=1)
y=df["label"]

smote=SMOTE(random_state=42)

X_res,y_res=smote.fit_resample(X,y)

print("Original:",y.value_counts())
print("Balanced:",pd.Series(y_res).value_counts())

ValueError: The target 'y' needs to have more than 1 class. Got 1 class instead

---
# 9️⃣ Model Training

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X_res,y_res,test_size=0.2,random_state=42)

scaler=StandardScaler()

X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

models={
"RandomForest":RandomForestClassifier(),
"XGBoost":xgb.XGBClassifier(eval_metric="logloss"),
"LightGBM":lgb.LGBMClassifier()
}

for name,model in models.items():

    model.fit(X_train,y_train)

    preds=model.predict(X_test)

    print(name)

    print("Accuracy:",accuracy_score(y_test,preds))

    print("F1:",f1_score(y_test,preds))

---
# 🔟 Inference on New Audio

In [ ]:
def predict_audio(file,model,scaler):

    y_clean,_,sr=preprocess_audio(file)

    feats=extract_audio_features(y_clean,sr)

    df=pd.DataFrame([feats])

    df=scaler.transform(df)

    pred=model.predict(df)

    print("Prediction:",pred)